# 04 · Re-write the submission with other cutoffs (no recomputation)

`03_full_lightgbm_submission.ipynb` saves the probability of every test pair in
`submission/test_probabilities/<country>.parquet`. This notebook turns those probabilities into a
new `matching_results.tsv` with other cutoffs per country, in a few minutes, so cutoffs can be
compared on the leaderboard without re-running 02 or 03.

**Why:** France has no labels, so its cutoff cannot be tuned on data. The leaderboard can do it.
Submit the file from 03 first, then one written here with a stricter France cutoff (e.g. 0.80).
The difference in the public score shows which way France should go. The public leaderboard uses
part of the test set, France included. Change only one country per submission, so the score
difference can be attributed.

**Same rules as 03:** each S2/S3 record goes to at most one S1 (the one with the highest
probability), only pairs with p ≥ the country's cutoff are kept, and every S1 of
`test_source1.tsv` gets exactly one row. `candidate_pairs.tsv` does not change, since the
candidates are the same. The files are checked against the submission rules at the end.

**Input:** the output of 03 (full) and the competition data. **Output:** `rethreshold/matching_results.tsv`.

In [ ]:
import glob
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd

# cutoffs to use; countries not listed keep the cutoff 03 tuned (printed below)
COUNTRY_THRESHOLDS = {"France": 0.80}


def search(patterns, roots):
    """First existing file matching any glob pattern under any root (shortest path wins); skips __MACOSX."""
    for root in roots:
        if root and os.path.isdir(root):
            hits = [h for pat in patterns for h in glob.glob(os.path.join(root, pat), recursive=True) if "__MACOSX" not in h]
            if hits:
                return Path(sorted(hits, key=len)[0])
    return None


prob_hit = search(["**/test_probabilities/*.parquet"], [os.environ.get("SUBMISSION_DIR", ""), "submission",
                                                          "/kaggle/working/submission", "/kaggle/input"])
assert prob_hit is not None, "test_probabilities/ not found: attach the output of 03 (full) or set SUBMISSION_DIR"
PROB_DIR = prob_hit.parent
CONFIG = json.loads((PROB_DIR.parent / "model" / "matcher_config.json").read_text(encoding="utf-8"))
TEST_S1_FILE = search(["**/test_source1.tsv"], [os.environ.get("DATA_ROOT", ""), "/kaggle/input",
                                                str(Path.cwd().parent / "Data" / "student_resource")])
assert TEST_S1_FILE is not None, "test_source1.tsv not found: attach the competition data or set DATA_ROOT"
OUT_DIR = Path(os.environ.get("RETHRESHOLD_DIR") or ("/kaggle/working/rethreshold" if os.path.isdir("/kaggle/working")
                                                     else "rethreshold"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
T_TUNED = CONFIG["threshold"]
print(f"probabilities: {PROB_DIR}\ntest S1: {TEST_S1_FILE}\ncutoff tuned by 03: {T_TUNED} | "
      f"cutoffs used by 03 per country: {CONFIG.get('country_thresholds') or 'none'}\nnew cutoffs: {COUNTRY_THRESHOLDS}")

In [ ]:
def one_owner(cand, p):
    """True for the rows where this S1 gives the candidate its highest p."""
    order = np.argsort(-p, kind="stable")
    first = ~pd.Series(cand[order]).duplicated().to_numpy()
    owner = np.zeros(len(p), bool)
    owner[order[first]] = True
    return owner


MATCHES, STATS = {}, []
for f in sorted(PROB_DIR.glob("*.parquet")):
    country = f.stem
    b = pd.read_parquet(f)
    t = COUNTRY_THRESHOLDS.get(country, T_TUNED)
    p = b["p"].to_numpy()
    keep = (p >= t) & one_owner(b["candidate_entity_id"].to_numpy(), p)
    kept = b.loc[keep, ["s1_entity_id", "candidate_entity_id", "p"]].sort_values(["s1_entity_id", "p"], ascending=[True, False])
    MATCHES.update(kept.groupby("s1_entity_id", sort=False)["candidate_entity_id"].agg(lambda s: ",".join(dict.fromkeys(s))).to_dict())
    n_s1 = b["s1_entity_id"].nunique()
    STATS.append({"country": country, "t": t, "S1": n_s1, "S1 with a predicted match": round(kept["s1_entity_id"].nunique() / n_s1, 4),
                  "matches per matched S1": round(len(kept) / max(kept["s1_entity_id"].nunique(), 1), 2)})
pd.DataFrame(STATS)

In [ ]:
with open(TEST_S1_FILE, encoding="utf-8") as fh:
    next(fh, None)
    TEST_S1 = list(dict.fromkeys(line.split("\t", 1)[0].strip() for line in fh if line.strip()))
path = OUT_DIR / "matching_results.tsv"
with open(path, "w", encoding="utf-8", newline="\n") as out:
    out.write("source1_entity_id\tmatched_entity_ids\n")
    for s in TEST_S1:
        out.write(f"{s}\t{MATCHES.get(s, '')}\n")

# checks: one row per test S1, no duplicate IDs in a list, S2/S3 IDs only, each S2/S3 used once
rows, ids_used, problems = [], [], []
with open(path, encoding="utf-8") as fh:
    assert fh.readline() == "source1_entity_id\tmatched_entity_ids\n"
    for line in fh:
        s1, _, rest = line.rstrip("\n").partition("\t")
        rows.append(s1)
        ids = rest.split(",") if rest else []
        if len(ids) != len(set(ids)):
            problems.append(f"duplicate ID in {s1}")
        ids_used += ids
if len(rows) != len(set(rows)) or set(rows) != set(TEST_S1):
    problems.append("rows do not match test_source1.tsv one-to-one")
if any(not i.startswith(("S2-", "S3-")) for i in ids_used):
    problems.append("an ID without S2-/S3- prefix")
if len(ids_used) != len(set(ids_used)):
    problems.append("an S2/S3 record is matched to more than one S1")
print(f"{path}: {len(rows):,} rows, {sum(1 for s in TEST_S1 if s in MATCHES):,} with matches |",
      "OK" if not problems else problems)
print("candidate_pairs.tsv is unchanged: submit this matching_results.tsv; run validate_submission.py on it with the "
      "candidate_pairs.tsv from 03.")